In [13]:
# Метакласс – это класс, экземплярами которого являются классы

# В  объектной  модели  Python  классы  являются  объектами,  поэтому  каждый класс должен быть экземпляром какого-то другого класса. 
# По умолчанию классы Python являются экземплярами класса type. Иными словами, type – метакласс для большинства встроенных и пользовательских классов:

# Эти классы не наследуют классу type. str и type – экземпляры type. И все они являются подклассами object

# Между  классами  object  и  type  имеется  удивительная  связь: 
# object – экземпляр type, а type – подкласс object. Эта связь «магическая»: выразить ее средствами Python невозможно, 
# потому что  любой  из  этих  классов  должен  существовать,  прежде  чем можно будет определить другой. И тот факт, что type является экземпляром самого себя, – тоже магия.

# Любой класс является экземпляром type, прямо или косвенно, но только метаклассы являются также подклассами type. Это самое главное, что нужно знать о метаклассах: 
# любой метакласс, в частности ABCMeta, наследует от type могущество, необходимое для конструирования классов.

# str.__class__
# <class 'type'>

# type.__class__
# <class 'type'>

In [ ]:
# Декораторы классов

# Проще, чем метаклассы, и меньше шансов на конфликты с базовыми классами и метаклассами.


# __set_name__
# Делает  лишним  привлечение  метаклассов  к  автоматическому  заданию имени  дескриптора

# __init_subclass__
# Предоставляет  способ  настройки  создания  класса,  прозрачный  для  конечного пользователя и даже более простой, чем декоратор. 
# Но при этом в сложной иерархии классов возможны конфликты.

In [12]:
# классом collections.Iterable является abc.ABCMeta. Класс Iterable абстрактный, а ABCMeta – нет

from collections.abc import Iterable
print(Iterable.__class__)

<class 'abc.ABCMeta'>


In [11]:
import abc
from abc import ABCMeta
print(ABCMeta.__class__)

<class 'type'>


In [15]:
# Тот же механизм работает на «метауровне», когда метакласс собирается создать новый экземпляр, т. е. класс. Рассмотрим следующее объявление:

class Klass(SuperKlass, metaclass=MetaKlass):
    x = 42
    def __init__(self, y):
        self.y = y

# Чтобы обработать предложение class, Python вызывает метод MetaKlass.__new__ с такими аргументами:

# meta_cls
# Сам метакласс (MetaKlass), потому что __new__ работает как метод класса.

# cls_name
# Строка Klass.

# bases
# Одноэлементный кортеж (SuperKlass,), который может иметь больше элементов в случае множественного наследования.

# cls_dict
# Отображение вида: {x: 42, `__init__`: <function __init__ at 0x1009c4040>}   
 

# При реализации MetaKlass.__new__ мы можем проинспектировать и изменить эти аргументы до передачи их методу super().__new__, который в конечном итоге вызовет type.__new__ для создания нового объекта класса.
# После возврата из super().__new__ мы можем дообработать вновь созданный класс, перед тем как возвращать его Python. Затем интерпретатор вызывает 
# SuperKlass.__init_subclass__, передавая созданный нами класс, после чего применяет к нему декоратор класса, если таковой задан. 
# Наконец, Python связывает объект класса с его именем в объемлющем пространстве имен – обычно это глобальное пространство имен модуля, если предложение class находится на верхнем уровне.
# Чаще всего в методе метакласса __new__ добавляются или заменяются элементы  отображения  cls_dict,  
# которое  представляет  пространство  имен  конструируемого класса. Например, перед вызовом super().__new__ мы можем внед-рить методы в конструируемый класс, 
# добавив функции в cls_dict. Заметим, однако, что добавлять методы можно и после построения класса, именно это и делают __init_subclass__ или декоратор класса. 
# Атрибут,  который  должен  быть  добавлен  в  cls_dict  до  вызова  type.__new__, – __slots__; этот вопрос обсуждался в разделе «Почему __init_subclass__ 
# не может конфигурировать __slots__» выше. Метод __new__  метакласса – идеальное место для конфигурирования __slots__. В следующем разделе объясняется, как это сделать     

NameError: name 'SuperKlass' is not defined

In [2]:
# __class__

# Атрибут экземпляра, который ссылается на его класс.

class Animal: pass
dog = Animal()
print(dog.__class__)

<class '__main__.Animal'>


In [3]:
# __name__

# Атрибут класса, функции, модуля или метода, который хранит его имя в виде строки.
# Где есть: У классов, функций, модулей. У экземпляров его нет (нужно обращаться через __class__.__name__).

def my_func(): pass
class MyClass: pass

print(MyClass.__name__)   # 'MyClass'
print(my_func.__name__)   # 'my_func'
print(__name__)           # '__main__' (имя текущего модуля)
print(dog.__class__.__name__)  # 'Animal'

MyClass
my_func
__main__
Animal


In [4]:
# __mro__

# (Method Resolution Order) Атрибут класса, который содержит кортеж классов в порядке, в котором Python ищет методы и атрибуты при наследовании.
# Где есть: Только у классов (объектов типа type).

class A: pass
class B(A): pass
class C(B): pass

print(C.__mro__)

(<class '__main__.C'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>)


In [11]:
# mro()

# Что это: Метод, возвращающий список классов в порядке разрешения методов (Method Resolution Order).
# Где есть: Только у классов.
# Используется функцией super() для поиска следующего класса в цепочке

class A: pass
class B(A): pass
class C(B): pass

print(C.mro())

[<class '__main__.C'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>]


In [10]:
# __bases__

# Кортеж прямых (непосредственных) родительских классов.
# Где есть: Только у классов.

class A: pass
class B: pass
class C(A, B): pass

print(C.__bases__)

(<class '__main__.A'>, <class '__main__.B'>)


In [8]:
# __subclasses__()

# Метод, возвращающий список прямых дочерних классов, которые уже загружены в память интерпретатора.
# Где есть: Только у классов.


class Base: pass
class Child1(Base): pass
class Child2(Base): pass

print(Base.__subclasses__())

[<class '__main__.Child1'>, <class '__main__.Child2'>]


In [9]:
# __qualname__

# Квалифицированное имя (qualified name). Строка, показывающая полный путь к классу/функции внутри модуля или вложенных классов.
# Где есть: У классов, функций, методов.

class Outer:
    class Inner: pass

print(Outer.__name__)        # 'Outer'
print(Outer.__qualname__)    # 'Outer'
print(Outer.Inner.__name__)   # 'Inner'
print(Outer.Inner.__qualname__) # 'Outer.Inner'

Outer
Outer
Inner
Outer.Inner


In [12]:
# type - Фабрика классов

# Класс type является метаклассом: класс, который строит классы. 
# Иными словами, экземплярами класса  type являются классы. В стандартной библиотеке есть и другие метаклассы, но type подразумевается по умолчанию

class MyClass(MySuperClass, MyMixin):
    x = 42
    def x2(self):
        return self.x * 2
    
    # эквивалентное создание через type с тремя аргументами
    
MyClass = type('MyClass',
               (MySuperClass, MyMixin),
               {'x': 42, 'x2': lambda self: self.x * 2},
          ) 

# name
# Идентификатор,  расположенный  после  ключевого  слова  class,  например MyClass.

# bases
# Кортеж  суперклассов,  расположенный  в  скобках  после  идентификатора класса, или (object,), если в предложении class нет суперклассов.

# dict
# Отображение имен атрибутов на значения. Вызываемые объекты становятся методами, другие значения становятся атрибутами класса


NameError: name 'MySuperClass' is not defined

In [13]:
from typing import Union, Any
from collections.abc import Iterable, Iterator

FieldNames = Union[str, Iterable[str]] # Пользователь может предоставить имена полей в виде одной строки или итерируемого объекта строк

def record_factory(cls_name: str, field_names: FieldNames) -> type[tuple]: # Принимаем такие же аргументы, как первые два в collections.namedtuple; возвращаем type, т. е. класс, который ведет себя как кортеж.

    slots = parse_identifiers(field_names) # Построить кортеж имен атрибутов, он станет атрибутом __slots__  нового класса.
    def __init__(self, *args, **kwargs) -> None: # Эта функция станет методом __init__  в новом классе. Она принимает позиционные и (или) именованные аргументы
        attrs = dict(zip(self.__slots__, args))
        attrs.update(kwargs)
        for name, value in attrs.items():
            setattr(self, name, value)

    def __iter__(self) -> Iterator[Any]: # Отдавать значения полей в порядке, определяемом атрибутом __slots_
        for name in self.__slots__:
            yield getattr(self, name)

    def __repr__(self): # Породить удобное представление, обходя __slots__ и self.
        values = ', '.join(f'{name}={value!r}'
            for name, value in zip(self.__slots__, self))
        cls_name = self.__class__.__name__
        return f'{cls_name}({values})'
    
    cls_attrs = dict( # Построить словарь атрибутов класса
        __slots__=slots,
        __init__=__init__,
        __iter__=__iter__,
        __repr__=__repr__,
    )
    return type(cls_name, (object,), cls_attrs) # Построить и вернуть новый класс, вызывая конструктор type

def parse_identifiers(names: FieldNames) -> tuple[str, ...]:
    if isinstance(names, str):
        names = names.replace(',', ' ').split() # Преобразовать строку names, в которой имена разделены пробелами или запятыми, в список строк.
    if not all(s.isidentifier() for s in names):
        raise ValueError('names must all be valid identifiers')
    return tuple(names)

In [14]:
Dog = record_factory('Dog', 'name weight owner')
Dog

__main__.Dog

In [15]:
rex = Dog('Rex', 30, 'Bob')
rex

Dog(name='Rex', weight=30, owner='Bob')

In [16]:
Dog.__mro__ 

(__main__.Dog, object)

In [3]:
# __init__subclass__

from collections.abc import Callable 
from typing import Any, NoReturn, get_type_hints

class Field:
    def __init__(self, name: str, constructor: Callable) -> None:
        if not callable(constructor) or constructor is type(None):
            raise TypeError(f'{name!r} type hint must be callable')
        self.name = name
        self.constructor = constructor

    def __set__(self, instance: Any, value: Any) -> None:
        if value is ...:
            value = self.constructor()
        else:
            try:
                value = self.constructor(value)
            except (TypeError, ValueError) as e:
                type_name = self.constructor.__name__
                msg = f'{value!r} is not compatible with {self.name}:{type_name}'
                raise TypeError(msg) from e
        instance.__dict__[self.name] = value

class Checked:
    @classmethod
    def _fields(cls) -> dict[str, type]:
        return get_type_hints(cls)

    def __init_subclass__(subclass) -> None: # __init_subclass__ вызывается, когда определяется подкласс текущего класса. Он получает новый подкласс в первом аргументе, именно поэтому я назвал аргумент subclass, а не cls, как обычно
        super().__init_subclass__() # Вызов super().__init_subclass__(), строго говоря, не является необходимым, но полезен для правильного взаимодействия с другими классами, которые реализовали .__init_subclass__() в том же графе наследования
        for name, constructor in subclass._fields().items(): # Обойти все поля name и constructor
            setattr(subclass, name, Field(name, constructor)) # создавая в subclass атрибут с данным именем name, связанный с дескриптором Field с параметрами name и constructor

    def __init__(self, **kwargs: Any) -> None:
        for name in self._fields():      # Для каждого поля класса name   
            value = kwargs.pop(name, ...)   # получить соответствующее значение value из kwargs и удалить его из kwargs. Использование ... (объект Ellipsis)  в качестве значения по умолчанию позволяет отличить заданные аргументы со значением None от незаданных    
            setattr(self, name, value)    
        if kwargs:     # Если в kwargs остались аргументы, то их имена не совпадают ни с одним из объявленных полей, и __init__ завершается ошибкой                       
            self.__flag_unknown_attrs(*kwargs)

    def __setattr__(self, name: str, value: Any) -> None: # Перехватывать все попытки установить атрибут экземпляра. Необходимо, чтобы предотвратить присваивание неизвестному атрибуту.
        if name in self._fields(): # Если  атрибут  с  именем  name  известен,  получить  соответствующий  дескриптор.
            cls = self.__class__
            descriptor = getattr(cls, name)
            descriptor.__set__(self, value) 
            # Обычно нам нет нужды вызывать метод __set__ дескриптора явно. 
            # Но в данном случае это необходимо, потому что __setattr__ перехватывает 
            # все попытки установить атрибут экземпляра, даже при наличии переопределяющего дескриптора типа Field
        else:
            self.__flag_unknown_attrs(name) # В противном случае атрибут с именем name неизвестен, и метод __flag_unknown_attrs возбуждает исключение.

    def __flag_unknown_attrs(self, *names: str) -> NoReturn: 
         # конструировать полезное сообщение об ошибке, 
         # содержащее все неожиданные аргументы, и возбудить исключение AttributeError. 
         # Это редкий пример специального типа NoReturn
        plural = 's' if len(names) > 1 else ''
        extra = ', '.join(f'{name!r}' for name in names)
        cls_name = repr(self.__class__.__name__)
        raise AttributeError(f'{cls_name} object has no attribute{plural} {extra}')
    
    def _asdict(self) -> dict[str, Any]: 
        # Создать словарь, содержащий атрибуты объекта Movie. 
        # Я предпочел бы назвать этот метод _as_dict, но решил последовать соглашению, 
        # заложенному методом _asdict в классе collections.namedtuple
        return {
            name: getattr(self, name)
            for name, attr in self.__class__.__dict__.items()
            if isinstance(attr, Field)
        }
    
    def __repr__(self) -> str: # Желание реализовать полезное представление в методе __repr__ – основная причина наличия метода _asdict в этом примере
        kwargs = ', '.join(
            f'{key}={value!r}' for key, value in self._asdict().items()
        )
        return f'{self.__class__.__name__}({kwargs})'        

In [16]:
# Этап импОрта и этап выполнения


# На этапе импорта интерпретатор:
# 1.  производит  синтаксический  анализ  исходного  кода  py-модуля  сверху вниз. Именно в это время могут возникать исключения SyntaxError;
# 2.  генерирует исполняемый байт-код;
# 3.  выполняет код верхнего уровня откомпилированного модуля.

# Если в локальном кеше __pycache__ существует актуальный pyc-файл, то этот этап пропускается, поскольку уже имеется готовый к выполнению байт-код.
# Хотя синтаксический анализ и компиляция, безусловно, являются действия-ми,  выполняемыми  на  «этапе  импорта»,  
# на  этой  стадии  могут  происходить и другие вещи, потому что почти каждое предложение в Python является исполняемым  в том  смысле,  что  в  нем  может  выполняться  пользовательский код, изменяющий состояние программы

# В частности, предложение import – не просто объявление1 оно еще и выполняет весь код, 
# находящийся на верхнем уровне импортируемого модуля, при первом его импорте в память процесса. 
# При последующих операциях импорта того же модуля используется кешированный код, так что происходит только связывание  имен.  
# Этот  верхнеуровневый  код  может делать  все,  что  угодно, включая такие типичные для «этапа выполнения» действия, как подключение к базе данных.

# Потому-то граница между «этапом импорта» и «этапом выполнения» размыта: 
# предложение import может активировать любые действия, которые принято считать частью «этапа выполнения». 
# И наоборот, «этап им-порта» может случиться глубоко внутри этапа выполнения, 
# потому что предложение  import  и  встроенная  функция  __import__  могут  употребляться  внутри любой обычной функции.



In [18]:
class MetaBunch(type): # Для создания нового метакласса унаследовать type.
    def __new__(meta_cls, cls_name, bases, cls_dict): 
        # __new__ работает, как метод класса, но класс является метаклассом, поэтому я назвал первый аргумент meta_cls
        # (часто употребляют также имя mcs). Остальные три аргумента такие же, как в трехаргументной сигнатуре для вызова type() с целью непосредственного создания класса.
        defaults = {} # В defaults будет храниться отображение имен атрибутов на их значения по умолчанию.
        def __init__(self, **kwargs): # Этот метод будет внедрен в новый класс
            for name, default in defaults.items(): # Прочитать defaults  и присвоить соответствующему атрибуту экземпляра значение, извлеченное из kwargs или подразумеваемое по умолчанию
                setattr(self, name, kwargs.pop(name, default))
            if kwargs:
                # Если в kwargs остались аргументы, значит, не нашлось слотов, в которые их можно было бы поместить. 
                # Мы полагаем, что быстрый отказ – правильный подход,  поэтому  не  хотим  молчаливо  игнорировать  лишние  элементы. 
                # Быстрое и эффективное решение – выбирать элементы из kwargs по одному и пытаться установить их в экземпляре, 
                # а если не получается, сразу возбуждать исключение AttributeError.
                extra = ', '.join(kwargs)
                raise AttributeError(f'No slots left for: {extra!r}')
            
        def __repr__(self):
            # __repr__ возвращает строку, которая выглядит как вызов конструктора, например Point(x=3). 
            # При этом именованные аргументы, принимающие значения по умолчанию, опускаются.
            rep = ', '.join(f'{name}={value!r}'
                            for name, default in defaults.items()
                            if (value := getattr(self, name)) != default)
            return f'{cls_name}({rep})'
        
        new_dict = dict(__slots__=[], __init__=__init__, __repr__=__repr__) # Инициализировать пространство имен для нового класса.
        for name, value in cls_dict.items(): # Обойти пространство имен пользовательского класса.
            if name.startswith('__') and name.endswith('__'):
                # Если  найдено  имя  name  с  двумя  подчерками,  копировать  элемент  в  пространство имен нового класса, 
                # если его там еще нет. Это не даст пользователю  перезаписать  __init__,  __repr__  и  другие  атрибуты,  
                # установленные самим Python, например __qualname__ и __module__.
                if name in new_dict:
                    raise AttributeError(f"Can't set {name!r} in {cls_name!r}")
                new_dict[name] = value
            else: # Если  имя  атрибута  name  не  начинается  двумя  подчерками,  добавить  его в конец __slots__ и сохранить его значение value в defaults.
                new_dict['__slots__'].append(name)
                defaults[name] = value
        return super().__new__(meta_cls, cls_name, bases, new_dict) # Построить и вернуть новый класс
    
class Bunch(metaclass=MetaBunch): # Предоставить базовый класс, чтобы пользователи не видели MetaBunch.
    pass